# Notebook 04 - Customer Segmentation
**Input:** `data/processed/customer_features.parquet` (5,265 customers x 38 columns)<br>
**Output:** `data/processed/customer_segments.parquet` - same customers, now with segment labels

## Why segmentation comes before predictive modeling

Predictive models (CLV, churn) tell us *what will happen*. Segmentation tells us *who these people are* - and that's what lets us act.

A segment is a label like "Champion" or "At Risk" that summarizes a customer's situation in one word.<br>
That label is what shows up in dashboards, in marketing automation rules, and in the conversation with your boss.<br>
"Predicted CLV of £247.30" is hard to act on; "Champion, predicted to stay" is immediately useful.

## Two complementary approaches
We'll do both because they tell different stories:

**1. RFM scoring (rules-based)** - Score each customer 1-5 on Recency, Frequency, Monetary using quintiles.<br>
Map score patterns to named segments (Champions, Loyal, At Risk, etc.). This is the classic marketing approach: explainable, no ML needed, defensible to any stakeholder.

**2. K-Means Clustering (data-dirven)** - Let an unsupervised algorithm find natural grouping using more features than just RFM. May surface segments rules would miss.

**Strategy:** Use RFM segment as the primary labels (because they're business-friendly), and use K-Means as a secondary cross-check to see if natural grouping agree or reveal something new.

## What this notebook is NOT

It's not modeling - there's no target variable, no train/test split, no predictions. Segmentation is **unsupervised**: we're organizing customers into groups based on their current state, not predicting their future.

## Setup


In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_style('whitegrid')

PROCESSED_DIR = Path('../data/processed')
REPORTS_DIR = Path('../reports/figures')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
features = pd.read_parquet(PROCESSED_DIR / 'customer_features.parquet')
print(f'Loaded {len(features):,} customers x {features.shape[1]} columns')

Loaded 5,256 customers x 38 columns
